In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


IMPORTING REQUIRED LIBRARIES


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, VGG16, MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd

Parameters

In [ ]:
batch_size = 32
epochs = 10
num_classes = 10
learning_rate = 0.0001
input_size = 224

Paths

In [ ]:
train_dir = "/content/drive/MyDrive/HA_EXP/train"
test_dir = "/content/drive/MyDrive/HA_EXP/test"
test_image_path = "/content/drive/MyDrive/HA_EXP/test/dogs/dog_114.jpg"

Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(input_size, input_size),
    batch_size=batch_size,
    class_mode='sparse'
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(input_size, input_size),
    batch_size=batch_size,
    class_mode='sparse'
)

class_names = list(train_generator.class_indices.keys())


Found 557 images belonging to 2 classes.
Found 140 images belonging to 2 classes.


COMMON MODEL FUNCTION

In [ ]:
def train_model(base_model_fn, model_name):
    base_model = base_model_fn(weights='imagenet',include_top=False,input_shape=(input_size, input_size, 3) )
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=outputs)
    for layer in base_model.layers:
        layer.trainable = False
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    print(f"\nTraining {model_name}...")
    model.fit(train_generator, epochs=epochs, validation_data=test_generator)
    return model


TRAIN ALL MODELS

In [ ]:
resnet_model = train_model(ResNet50, "ResNet50")
vgg_model = train_model(VGG16, "VGG16")
mobilenet_model = train_model(MobileNetV2, "MobileNetV2")


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training ResNet50...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 194s 10s/step - accuracy: 0.1896 - loss: 2.3670 - val_accuracy: 0.5071 - val_loss: 0.7474
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 14s 814ms/step - accuracy: 0.5357 - loss: 0.7202 - val_accuracy: 0.5714 - val_loss: 0.6927
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 14s 756ms/step - accuracy: 0.5722 - loss: 0.6874 - val_accuracy: 0.4929 - val_loss: 0.7093
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 13s 741ms/step - accuracy: 0.5228 - loss: 0.6872 - val_accuracy: 0.5214 - val_loss: 0.7033
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 13s 718ms/step - accuracy: 0.5438 - loss: 0.6871 - val_accuracy: 0.5143 - val_loss: 0.6896
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 13s 718ms/step - accuracy: 0.5588 - loss: 0.6984 - val_accuracy: 0.5286 - val_loss: 0.6844
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 14s 750ms/step - accuracy: 0.5896 - loss: 0.6714 - val_accuracy: 0.5714 - val_loss: 0.6783
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 13s 726ms/step - accuracy: 0.5445 - loss: 0.6854 - val_accur

IMAGE PREDICTION FUNCTION

In [ ]:
def predict_image(image_path, model):
    img = image.load_img(image_path, target_size=(input_size, input_size))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)
    cls = np.argmax(preds[0])
    conf = preds[0][cls]

    return class_names[cls], float(conf)


COMPARISON TABLE

In [ ]:
models = {
    "ResNet50": resnet_model,
    "VGG16": vgg_model,
    "MobileNetV2": mobilenet_model
}

results = []

for name, mdl in models.items():
    label, conf = predict_image(test_image_path, mdl)
    results.append([name, label, round(conf, 4)])

comparison_table = pd.DataFrame(
    results,
    columns=["Model", "Predicted Label", "Confidence"]
)

comparison_table


1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step


,Model,Predicted Label,Confidence
0,ResNet50,dogs,0.5920
1,VGG16,dogs,0.6246
2,MobileNetV2,dogs,0.9997
